# 📈 Equity Analysis — Demo Notebook

This notebook is a **thin presentation layer** over the `stockanalysis` package.
All logic lives in the package (`src/stockanalysis/`); here we just call it and
render the results interactively.

Run **all cells top to bottom**. Requires live network access (Yahoo Finance).

## 0 · Setup

Make the package importable (editable install is preferred — see README — but this also works in-place) and enable inline Plotly.

In [1]:
import sys, logging
sys.path.insert(0, "../src")          # run without installing; or: pip install -e ..

import plotly.io as pio
pio.renderers.default = "notebook"     # inline interactive charts
logging.basicConfig(level=logging.INFO, format="%(message)s")

from pathlib import Path
import pandas as pd
import stockanalysis as sa
print("stockanalysis", sa.__version__)

stockanalysis 0.1.0


## 1 · Run the pipeline

`run()` does ingest → screen → indicators → signals and returns a `Results` object.

In [2]:
results = sa.run(out_dir="../output")   # report.html + everything else land in the same run folder by default
screened_df   = results.screened_df
signal_matrix = results.signal_matrix
tech          = results.tech
out = Path(results.run_dir)            # this run's timestamped folder — §4/§5 both write here
out.mkdir(parents=True, exist_ok=True)
print(f"{len(signal_matrix)} signals · {len(tech)} tickers with indicators · output → {out}/")

Fetching data for 18 tickers (hits Yahoo Finance once per ticker)...


  AAPL      753 rows (2023-08-21 -> 2026-08-20)


  MSFT      753 rows (2023-08-21 -> 2026-08-20)


  NVDA      753 rows (2023-08-21 -> 2026-08-20)


  GOOGL     753 rows (2023-08-21 -> 2026-08-20)


...

  MRVL      753 rows (2023-08-21 -> 2026-08-20)


  CRM       753 rows (2023-08-21 -> 2026-08-20)


Ingestion complete: 18/18 tickers with price history.


Saved combined report to '../output/2026-08-20_225300/report.html'.


18 signals · 18 tickers with indicators · output → ../output/2026-08-20_225300/


## 2 · Screener (styled)

In [3]:
display_cols = ["Sector","PE","EPS_Growth","Rev_Growth","Debt_Equity","Div_Yield","FCF","Fundamental_Score"]
def style_screen(df):
    if df.empty: return df
    return (df[display_cols].style
        .format({"PE":"{:.1f}","EPS_Growth":"{:.1%}","Rev_Growth":"{:.1%}",
                 "Div_Yield":"{:.2%}","Debt_Equity":"{:.2f}","FCF":"{:,.0f}"}, na_rep="—")
        .background_gradient(subset=["Fundamental_Score"], cmap="Greens", vmin=0, vmax=6)
        .set_caption("Fundamental Screener — sorted by Score (0–6)"))
style_screen(screened_df)

,Sector,PE,EPS_Growth,Rev_Growth,Debt_Equity,Div_Yield,FCF,Fundamental_Score
Ticker,,,,,,,,
GOOGL,Technology,17.3,294.0%,24.2%,0.19,26.00%,"22,665,000,960",6
MU,Technology,21.2,1368.5%,345.7%,0.06,6.00%,"7,639,499,776",6
BRK-B,Financials,12.6,107.8%,10.0%,0.17,—,"71,975,247,872",5
META,Technology,20.6,-13.4%,28.0%,0.43,38.00%,"21,553,625,088",5
AMZN,Consumer Discretionary,20.9,242.3%,19.6%,0.46,—,"3,219,124,992",5
CRM,Technology,23.9,52.2%,13.3%,1.24,85.00%,"16,552,999,936",5
MSFT,Technology,27.0,31.7%,17.7%,0.29,75.00%,"16,545,500,160",5
NVDA,Technology,33.3,214.5%,85.2%,0.07,46.00%,"46,335,873,024",5
AAPL,Technology,36.4,28.7%,16.4%,0.78,34.00%,"107,721,875,456",5


## 3 · Signal matrix (styled)

**How the scores combine.** Each row fuses two scores into the final action:

- **Fundamental score (0–6):** one point per screener threshold passed.
- **Technical score (0–7, registry-driven):** +1 for each of — price > EMA50, RSI 35–70, a recent bullish MACD crossover, a positive close-regression slope, a **rising EMA50**, volume confirmation, and **price near the lower EMA envelope** (bottom 25% of the band — a mean-reversion entry). The components live in `stockanalysis.signals.TECHNICAL_COMPONENTS`; the max equals its length, so adding/removing a component rescales automatically.
- **Composite** = `0.70·(fund/6) + 0.30·(tech/N)` → **Buy ≥ 0.60 · Hold ≥ 0.40 · Watch** otherwise.
- **Posture** auto-scales with the component count, from two mirrored fractions: **Bullish** at score ≥ ⅔·max (≥ 5 of 7), **Bearish** at score ≤ ⅓·max (≤ 2 of 7), else **Neutral**. Note the score only counts *bullish* confirmations, so a low score means "nothing confirming", not an explicit sell signal.

In [4]:
from stockanalysis.signals import TECHNICAL_COMPONENTS
def style_signals(df):
    if df.empty: return df
    ac = {"Buy":"#1b7837","Hold":"#b8860b","Watch":"#8c8c8c"}
    pc = {"Bullish":"#1b7837","Neutral":"#b8860b","Bearish":"#b2182b"}
    cols = ["Ticker","Sector","Fundamental Score","Technical Posture","Tech Score","Composite","Final Action Signal"]
    return (df[cols].style
        .map(lambda v: f"color:white;font-weight:700;background-color:{ac.get(v,'#8c8c8c')}", subset=["Final Action Signal"])
        .map(lambda v: f"color:{pc.get(v,'#333')};font-weight:600", subset=["Technical Posture"])
        .background_gradient(subset=["Fundamental Score"], cmap="Greens", vmin=0, vmax=6)
        .background_gradient(subset=["Tech Score"], cmap="Greens", vmin=0, vmax=len(TECHNICAL_COMPONENTS))
        .background_gradient(subset=["Composite"], cmap="RdYlGn", vmin=0, vmax=1)
        .format({"Composite":"{:.2f}"}).set_properties(**{"text-align":"center"})
        .set_caption("🎯 Combined Signal Matrix"))
style_signals(signal_matrix)

,Ticker,Sector,Fundamental Score,Technical Posture,Tech Score,Composite,Final Action Signal
0,MU,Technology,6,Neutral,4,0.87,Buy
1,GOOGL,Technology,6,Neutral,3,0.83,Buy
2,BRK-B,Financials,5,Bullish,5,0.80,Buy
3,MSFT,Technology,5,Bullish,5,0.80,Buy
4,NVDA,Technology,5,Bullish,5,0.80,Buy
5,CRM,Technology,5,Neutral,4,0.76,Buy
6,AAPL,Technology,5,Neutral,4,0.76,Buy
7,ASML,Technology,5,Neutral,4,0.76,Buy
8,META,Technology,5,Neutral,3,0.71,Buy
9,AMZN,Consumer Discretionary,5,Neutral,3,0.71,Buy


## 4 · Combined report — top 5 picks

One self-contained `report.html` bundles everything: the Fundamental Screener (§2) and Combined Signal Matrix (§3) tables in full, plus top-`TOP_N` Technical Dashboards, top-`TOP_N` Fundamental Profiles, and the Daily Market Overview chart (index performance + VIX). Only the dashboards/profiles sections are capped to `TOP_N` — the screener/signal-matrix tables always show every ticker.

`sa.report.build_full_report` (pure — data in, HTML string out) + `sa.report.save_report` (thin I/O wrapper) are the exact functions `sa.run()`'s default `save_report=True` already calls; this cell makes the call explicit so the intermediate `profiles`/`overview_data` stay inspectable, and previews the saved file inline.

Headless equivalent: `sa.run(out_dir="../output")` — the report is on by default (`--no-report` / `save_report=False` opts out; `--top` / `top_n` controls the cap, default 5).

In [5]:
from datetime import datetime
from IPython.display import IFrame

TOP_N = 5
tickers = sa.signals.top_tickers(signal_matrix, TOP_N) or list(tech)[:TOP_N]
display(style_signals(signal_matrix.head(TOP_N)))

profiles = [sa.profile.build_profile(t, screened_df) for t in tickers]
overview_data = sa.overview.daily_overview(signal_matrix=signal_matrix, tech=tech)

html_doc = sa.report.build_full_report(
    screened_df, signal_matrix, tech, profiles, overview_data,
    selected=tickers, generated_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
)
report_path = sa.report.save_report(html_doc, out / "report.html")
print(f"saved combined report → {report_path}")
IFrame(report_path, width="100%", height=900)

,Ticker,Sector,Fundamental Score,Technical Posture,Tech Score,Composite,Final Action Signal
0,MU,Technology,6,Neutral,4,0.87,Buy
1,GOOGL,Technology,6,Neutral,3,0.83,Buy
2,BRK-B,Financials,5,Bullish,5,0.80,Buy
3,MSFT,Technology,5,Bullish,5,0.80,Buy
4,NVDA,Technology,5,Bullish,5,0.80,Buy


saved combined report → ../output/2026-08-20_225300/report.html


## 5 · Export

Persist via the pluggable exporters (Excel works out of the box; Google Sheets needs the `[gsheets]` extra + credentials — see README). This lands next to `report.html` from §4, in the same run folder `../output/<timestamp>/`.

In [6]:
sa.outputs.get_exporter("excel", path=str(out / "signal_matrix.xlsx")).export(signal_matrix, screened_df)
print("run folder now holds:", ", ".join(sorted(p.name for p in out.iterdir())))
# sa.outputs.get_exporter("gsheets", spreadsheet="<id|name>").export(signal_matrix, screened_df)

Exported results to '../output/2026-08-20_225300/signal_matrix.xlsx' (sheets: Signal Matrix, Fundamentals).


run folder now holds: report.html, signal_matrix.xlsx
